<a href="https://colab.research.google.com/github/Delean-Mafra/faculdade/blob/main/Arquitetura-de-BI-e-Big-Data-com-Ciencia-de-Dados-Aplicada-main/Atividade_Pratica_14_DBDBD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install elasticsearch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.3/66.3 kB 2.7 MB/s eta 0:00:00


In [2]:
import os

# Insira isso antes de iniciar o serviço do Elasticsearch
os.system('echo "xpack.ml.enabled: false" | sudo tee -a /etc/elasticsearch/elasticsearch.yml')
os.system('echo "xpack.security.enabled: false" | sudo tee -a /etc/elasticsearch/elasticsearch.yml')

256

In [3]:
!echo "discovery.type: single-node" | sudo tee -a /etc/elasticsearch/elasticsearch.yml
!echo "xpack.security.enabled: false" | sudo tee -a /etc/elasticsearch/elasticsearch.yml
!echo "xpack.ml.enabled: false" | sudo tee -a /etc/elasticsearch/elasticsearch.yml
!chown -R daemon:daemon /usr/share/elasticsearch /var/lib/elasticsearch /var/log/elasticsearch /etc/elasticsearch
!sudo -u daemon ES_JAVA_OPTS="-Xms512m -Xmx512m" /usr/share/elasticsearch/bin/elasticsearch -d
!sleep 25

tee: /etc/elasticsearch/elasticsearch.yml: No such file or directory
discovery.type: single-node
tee: /etc/elasticsearch/elasticsearch.yml: No such file or directory
xpack.security.enabled: false
tee: /etc/elasticsearch/elasticsearch.yml: No such file or directory
xpack.ml.enabled: false
chown: cannot access '/usr/share/elasticsearch': No such file or directory
chown: cannot access '/var/lib/elasticsearch': No such file or directory
chown: cannot access '/var/log/elasticsearch': No such file or directory
chown: cannot access '/etc/elasticsearch': No such file or directory
sudo: /usr/share/elasticsearch/bin/elasticsearch: command not found


In [4]:
import os
import time

os.system('pkill -f elasticsearch')
time.sleep(2)

os.system('echo "discovery.type: single-node" > /etc/elasticsearch/elasticsearch.yml')
os.system('echo "xpack.security.enabled: false" >> /etc/elasticsearch/elasticsearch.yml')
os.system('echo "xpack.ml.enabled: false" >> /etc/elasticsearch/elasticsearch.yml')
os.system('echo "ingest.geoip.downloader.enabled: false" >> /etc/elasticsearch/elasticsearch.yml')
os.system('chown -R daemon:daemon /usr/share/elasticsearch /var/lib/elasticsearch /var/log/elasticsearch /etc/elasticsearch')

os.system('sudo -u daemon ES_JAVA_OPTS="-Xms400m -Xmx400m" /usr/share/elasticsearch/bin/elasticsearch -d')

print("Aguardando inicialização do Elasticsearch...")
for i in range(30):
    time.sleep(1)
    if os.system('curl -s http://localhost:9200 > /dev/null') == 0:
        print("Elasticsearch online e pronto para uso!")
        break
else:
    print("Falha ao iniciar. Verifique o motivo exato no log abaixo:\n")
    os.system('cat /var/log/elasticsearch/elasticsearch.log | tail -n 50')

Aguardando inicialização do Elasticsearch...
Falha ao iniciar. Verifique o motivo exato no log abaixo:



In [5]:
import os
import time

# 1. Configurações essenciais para evitar crash no ambiente do Colab
os.system('echo "discovery.type: single-node" | sudo tee -a /etc/elasticsearch/elasticsearch.yml')
os.system('echo "xpack.security.enabled: false" | sudo tee -a /etc/elasticsearch/elasticsearch.yml')
os.system('echo "xpack.ml.enabled: false" | sudo tee -a /etc/elasticsearch/elasticsearch.yml')

# 2. Ajuste de permissões
# O Colab roda como root, mas o Elasticsearch proíbe sua execução como root.
# Mudamos o dono dos diretórios para o usuário restrito 'daemon'.
os.system('chown -R daemon:daemon /usr/share/elasticsearch /var/lib/elasticsearch /var/log/elasticsearch /etc/elasticsearch')

# 3. Inicia o Elasticsearch forçando o usuário daemon e limitando a memória RAM (Java Heap) para 512MB
print("Iniciando o servidor Elasticsearch em background...")
os.system('sudo -u daemon ES_JAVA_OPTS="-Xms512m -Xmx512m" /usr/share/elasticsearch/bin/elasticsearch -d')

# 4. Aguarda o serviço subir completamente na porta 9200 antes de prosseguir
print("Aguardando 25 segundos para o servidor ficar online...")
time.sleep(25)

Iniciando o servidor Elasticsearch em background...
Aguardando 25 segundos para o servidor ficar online...


In [6]:
!cat /var/log/elasticsearch/elasticsearch.log | tail -n 30

cat: /var/log/elasticsearch/elasticsearch.log: No such file or directory


In [7]:
%%bash
# 1. Matar qualquer processo travado do Elasticsearch
pkill -f elasticsearch
sleep 2

# 2. Limpar dados antigos/bloqueados que impedem a inicialização
rm -rf /var/lib/elasticsearch/*

# 3. Criar usuário dedicado (o Colab usa root por padrão, o que quebra o ES)
useradd -m elasticuser 2>/dev/null || true

# 4. Criar um arquivo de configuração limpo e minimalista
cat <<EOF > /etc/elasticsearch/elasticsearch.yml
cluster.name: colab-cluster
node.name: colab-node
path.data: /var/lib/elasticsearch
path.logs: /var/log/elasticsearch
network.host: 127.0.0.1
http.port: 9200
discovery.type: single-node
xpack.security.enabled: false
xpack.ml.enabled: false
ingest.geoip.downloader.enabled: false
EOF

# 5. Aplicar permissões rigorosas para o novo usuário
chown -R elasticuser:elasticuser /usr/share/elasticsearch /var/lib/elasticsearch /var/log/elasticsearch /etc/elasticsearch

# 6. Iniciar o serviço em background com memória limitada
echo "Iniciando Elasticsearch..."
sudo -u elasticuser ES_JAVA_OPTS="-Xms400m -Xmx400m" /usr/share/elasticsearch/bin/elasticsearch -d

# 7. Monitorar a porta 9200 para confirmar o sucesso (timeout de 35 segundos)
for i in {1..35}; do
  if curl -s http://localhost:9200 > /dev/null; then
    echo "Elasticsearch está ONLINE e pronto para receber conexões!"
    exit 0
  fi
  sleep 1
done

echo "Falha ao iniciar. Últimas linhas do log de erro:"
cat /var/log/elasticsearch/colab-cluster.log | tail -n 40

Iniciando Elasticsearch...
Falha ao iniciar. Últimas linhas do log de erro:


bash: line 12: /etc/elasticsearch/elasticsearch.yml: No such file or directory
chown: cannot access '/usr/share/elasticsearch': No such file or directory
chown: cannot access '/var/lib/elasticsearch': No such file or directory
chown: cannot access '/var/log/elasticsearch': No such file or directory
chown: cannot access '/etc/elasticsearch': No such file or directory
sudo: /usr/share/elasticsearch/bin/elasticsearch: command not found
cat: /var/log/elasticsearch/colab-cluster.log: No such file or directory


In [8]:
!pip uninstall -yq elastic-transport elasticsearch
!pip install -q pyspark prometheus_client elasticsearch==7.17.9 "urllib3<2.0.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.0/386.0 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 12.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
blobfile 3.3.0 requires urllib3>=2, but you have urllib3 1.26.20 which is incompatible.


In [10]:
%%bash
# 1. Matar processos travados e remover a instalação com erro do apt
pkill -f elasticsearch
apt-get remove --purge -y elasticsearch > /dev/null 2>&1
rm -rf /content/elasticsearch-*

# 2. Baixar os binários standalone da versão mais recente (resolve o bug do Java Security)
cd /content
echo "Baixando Elasticsearch..."
wget -q https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-8.15.0-linux-x86_64.tar.gz
tar -xzf elasticsearch-8.15.0-linux-x86_64.tar.gz

# 3. Aplicar configurações limpas para o Colab
cat <<EOF > /content/elasticsearch-8.15.0/config/elasticsearch.yml
cluster.name: colab-cluster
node.name: colab-node
network.host: 127.0.0.1
http.port: 9200
discovery.type: single-node
xpack.security.enabled: false
xpack.ml.enabled: false
EOF

# 4. Dar permissão total ao usuário daemon apenas nesta pasta
chown -R daemon:daemon /content/elasticsearch-8.15.0

# 5. Iniciar o serviço em background
echo "Iniciando Elasticsearch..."
sudo -u daemon ES_JAVA_OPTS="-Xms400m -Xmx400m" /content/elasticsearch-8.15.0/bin/elasticsearch -d

# 6. Monitorar a porta 9200 para confirmação (timeout de 45 segundos)
for i in {1..45}; do
  if curl -s http://localhost:9200 > /dev/null; then
    echo "Elasticsearch está ONLINE e pronto para uso!"
    exit 0
  fi
  sleep 1
done

echo "Falha ao iniciar. Últimas linhas do log:"
cat /content/elasticsearch-8.15.0/logs/colab-cluster.log | tail -n 30

Baixando Elasticsearch...
Iniciando Elasticsearch...
[2026-09-05T18:46:48,194][INFO ][o.e.n.NativeAccess       ] [colab-node] Using native vector library; to disable start with -Dorg.elasticsearch.nativeaccess.enableVectorLibrary=false
[2026-09-05T18:46:49,317][INFO ][o.e.n.NativeAccess       ] [colab-node] Using [jdk] native provider and native methods for [Linux]
[2026-09-05T18:46:50,056][INFO ][o.a.l.i.v.PanamaVectorizationProvider] [colab-node] Java vector incubator API enabled; uses preferredBitSize=256; FMA enabled
[2026-09-05T18:46:52,084][INFO ][o.e.n.Node               ] [colab-node] version[8.15.0], pid[3868], build[tar/1a77947f34deddb41af25e6f0ddb8e830159c179/2024-08-05T10:05:34.233336849Z], OS[Linux/6.6.122+/amd64], JVM[Oracle Corporation/OpenJDK 64-Bit Server VM/22.0.1/22.0.1+8-16]
[2026-09-05T18:46:52,087][INFO ][o.e.n.Node               ] [colab-node] JVM home [/content/elasticsearch-8.15.0/jdk], using bundled JDK [true]
[2026-09-05T18:46:52,088][INFO ][o.e.n.Node       

Sep 05, 2026 6:46:46 PM sun.util.locale.provider.LocaleProviderAdapter <clinit>
ERROR: Elasticsearch did not exit normally - check the logs at /content/elasticsearch-8.15.0/logs/colab-cluster.log

ERROR: Elasticsearch died while starting up, with exit code 1


In [12]:
import random
import csv
from datetime import datetime, timedelta
import os

# Sementes para reprodutibilidade
random.seed(42)

# Categorias e nomes de produtos
CATEGORIAS = ['Eletrônicos', 'Roupas', 'Casa e Decoração', 'Livros', 'Brinquedos', 'Esportes', 'Beleza', 'Automotivo']
PRODUTOS_POR_CATEGORIA = {
    'Eletrônicos': ['Smartphone', 'Notebook', 'Tablet', 'Fone de Ouvido', 'Carregador', 'Monitor', 'Teclado', 'Mouse'],
    'Roupas': ['Camiseta', 'Calça Jeans', 'Vestido', 'Jaqueta', 'Sapato', 'Bolsa', 'Cinto'],
    'Casa e Decoração': ['Sofá', 'Mesa', 'Cadeira', 'Estante', 'Luminária', 'Quadro', 'Tapete'],
    'Livros': ['Romance', 'Ficção Científica', 'Biografia', 'Autoajuda', 'Fantasia', 'Suspense'],
    'Brinquedos': ['Boneca', 'Carrinho', 'Quebra-cabeça', 'Jogo de Tabuleiro', 'Pelúcia'],
    'Esportes': ['Bola', 'Tênis', 'Raquete', 'Bicicleta', 'Halter'],
    'Beleza': ['Shampoo', 'Condicionador', 'Base', 'Máscara Facial', 'Perfume'],
    'Automotivo': ['Pneu', 'Bateria', 'Óleo', 'Filtro', 'Palheta']
}

# Função para gerar nome de produto
def gerar_nome_produto(categoria):
    return random.choice(PRODUTOS_POR_CATEGORIA[categoria]) + ' ' + str(random.randint(1, 100))

# Geração de produtos
def gerar_produtos(qtd=10000):
    produtos = []
    for i in range(1, qtd+1):
        categoria = random.choice(CATEGORIAS)
        nome = gerar_nome_produto(categoria)
        preco = round(random.uniform(10.0, 5000.0), 2)
        descricao = f'{nome} de alta qualidade, ideal para uso diário.'
        estoque = random.randint(0, 100)
        produtos.append([i, nome, categoria, preco, descricao, estoque])
    return produtos

# Geração de avaliações
def gerar_avaliacoes(produtos, qtd=50000):
    avaliacoes = []
    ids_produtos = [p[0] for p in produtos]
    for i in range(1, qtd+1):
        id_prod = random.choice(ids_produtos)
        nota = random.randint(1, 5)
        comentario = random.choice(['Ótimo!', 'Bom produto.', 'Poderia ser melhor.', 'Excelente!', 'Não gostei.'])
        data = datetime.now() - timedelta(days=random.randint(1, 730))
        avaliacoes.append([i, id_prod, nota, comentario, data.strftime('%Y-%m-%d')])
    return avaliacoes

# Geração de transações
def gerar_transacoes(produtos, qtd=100000):
    transacoes = []
    ids_produtos = [p[0] for p in produtos]
    for i in range(1, qtd+1):
        id_prod = random.choice(ids_produtos)
        preco_unit = next(p[3] for p in produtos if p[0] == id_prod)
        quantidade = random.randint(1, 5)
        data = datetime.now() - timedelta(days=random.randint(1, 365))
        id_cliente = random.randint(1, 5000)
        transacoes.append([i, id_prod, quantidade, preco_unit, data.strftime('%Y-%m-%d'), id_cliente])
    return transacoes

# Salvar dados em arquivos .txt (CSV)
def salvar_dados(dados, nome_arquivo, cabecalho):
    with open(nome_arquivo, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f, delimiter=';')
        writer.writerow(cabecalho)
        writer.writerows(dados)

# Criar diretório para dados
os.makedirs('dados', exist_ok=True)

# Gerar e salvar
produtos = gerar_produtos(10000)
salvar_dados(produtos, 'dados/produtos.txt', ['id', 'nome', 'categoria', 'preco', 'descricao', 'estoque'])

avaliacoes = gerar_avaliacoes(produtos, 50000)
salvar_dados(avaliacoes, 'dados/avaliacoes.txt', ['id', 'id_produto', 'nota', 'comentario', 'data'])

transacoes = gerar_transacoes(produtos, 100000)
salvar_dados(transacoes, 'dados/transacoes.txt', ['id', 'id_produto', 'quantidade', 'preco_unitario', 'data', 'id_cliente'])

# Particionamento por categoria (sharding) - exemplo para produtos
for categoria in CATEGORIAS:
    produtos_cat = [p for p in produtos if p[2] == categoria]
    if produtos_cat:
        salvar_dados(produtos_cat, f'dados/produtos_{categoria}.txt', ['id', 'nome', 'categoria', 'preco', 'descricao', 'estoque'])

# Particionamento por mês (sharding) - exemplo para transações
from collections import defaultdict
transacoes_por_mes = defaultdict(list)
for t in transacoes:
    mes = t[4][:7]  # ano-mês
    transacoes_por_mes[mes].append(t)

for mes, lista in transacoes_por_mes.items():
    salvar_dados(lista, f'dados/transacoes_{mes}.txt', ['id', 'id_produto', 'quantidade', 'preco_unitario', 'data', 'id_cliente'])

print("Dados gerados com sucesso!")

Dados gerados com sucesso!


In [14]:
from elasticsearch import Elasticsearch, helpers
import json

# Conectar ao Elasticsearch (localhost:9200)
es = Elasticsearch([{'host': 'localhost', 'port': 9200}])

if es.ping():
    print("Conectado ao Elasticsearch")
    # Criar índice
    es.indices.create(index='produtos', ignore=400)
    # Indexar produtos (usando bulk)
    actions = []
    for p in produtos:
        doc = {
            '_index': 'produtos',
            '_id': p[0],
            '_source': {
                'nome': p[1],
                'categoria': p[2],
                'preco': p[3],
                'descricao': p[4],
                'estoque': p[5]
            }
        }
        actions.append(doc)
    helpers.bulk(es, actions)
    print("Produtos indexados.")

    # Busca por "Smartphone" na categoria "Eletrônicos"
    res = es.search(index='produtos', body={
        "query": {
            "bool": {
                "must": [
                    {"match": {"nome": "Smartphone"}},
                    {"term": {"categoria": "Eletrônicos"}}
                ]
            }
        }
    })
    print("Resultados da busca:")
    for hit in res['hits']['hits']:
        print(hit['_source'])
else:
    print("Elasticsearch não disponível. Usando simulação em memória.")

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/urllib3/connection.py", line 174, in _new_conn
    conn = connection.create_connection(
        (self._dns_host, self.port), self.timeout, **extra_kw
    )
  File "/usr/local/lib/python3.13/dist-packages/urllib3/util/connection.py", line 95, in create_connection
    raise err
  File "/usr/local/lib/python3.13/dist-packages/urllib3/util/connection.py", line 85, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
ConnectionRefusedError: [Errno 111] Connection refused

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/elasticsearch/connection/http_urllib3.py", line 255, in perform_request
    response = self.pool.urlopen(
        method, url, body, retries=Retry(False), headers=request_headers, **kw
    )
  File "/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py", line 802, in url

Elasticsearch não disponível. Usando simulação em memória.


In [15]:
import pandas as pd
import glob

# Lê todos os arquivos de transações particionados por mês
arquivos = glob.glob('dados/transacoes_*.txt')
dfs = []
for arq in arquivos:
    df = pd.read_csv(arq, delimiter=';')
    dfs.append(df)
df_transacoes = pd.concat(dfs, ignore_index=True)

# Lê produtos
df_produtos = pd.read_csv('dados/produtos.txt', delimiter=';')

# Junta as tabelas
df_vendas = df_transacoes.merge(df_produtos[['id', 'categoria']], left_on='id_produto', right_on='id')
df_vendas['faturamento'] = df_vendas['quantidade'] * df_vendas['preco_unitario']
df_vendas['mes'] = pd.to_datetime(df_vendas['data']).dt.to_period('M')

# Agregação por categoria e mês
agregado = df_vendas.groupby(['categoria', 'mes']).agg(
    total_vendas=('faturamento', 'sum'),
    qtde_itens=('quantidade', 'sum')
).reset_index()

print(agregado.head(10))

# Salvar resultado
agregado.to_csv('dados/analise_vendas.txt', sep=';', index=False)

    categoria      mes  total_vendas  qtde_itens
0  Automotivo  2025-09    6993214.41        2795
1  Automotivo  2025-10    7746701.96        3178
2  Automotivo  2025-11    7435670.26        3033
3  Automotivo  2025-12    7355819.91        3115
4  Automotivo  2026-01    7476706.75        3153
5  Automotivo  2026-02    6743753.94        2805
6  Automotivo  2026-03    7714916.64        3169
7  Automotivo  2026-04    8018068.66        3278
8  Automotivo  2026-05    7689939.29        3069
9  Automotivo  2026-06    7311527.39        3031


In [16]:
from collections import defaultdict

# Carregar transações
df_transacoes = pd.read_csv('dados/transacoes.txt', delimiter=';')

# Criar lista de compras por cliente
compras_cliente = defaultdict(list)
for _, row in df_transacoes.iterrows():
    compras_cliente[row['id_cliente']].append(row['id_produto'])

# Calcular co-ocorrência de produtos (quem comprou A também comprou B)
co_ocorrencia = defaultdict(lambda: defaultdict(int))
for produtos_comprados in compras_cliente.values():
    for i in range(len(produtos_comprados)):
        for j in range(i+1, len(produtos_comprados)):
            a, b = produtos_comprados[i], produtos_comprados[j]
            co_ocorrencia[a][b] += 1
            co_ocorrencia[b][a] += 1

# Função de recomendação para um cliente
def recomendar_para_cliente(id_cliente, top_n=5):
    if id_cliente not in compras_cliente:
        return []
    historico = compras_cliente[id_cliente]
    # Soma pontuações dos produtos co-ocorrentes
    scores = defaultdict(float)
    for item in historico:
        for outro, contagem in co_ocorrencia[item].items():
            if outro not in historico:  # não recomendar já comprados
                scores[outro] += contagem
    # Ordenar
    recomendados = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [prod_id for prod_id, score in recomendados[:top_n]]

# Exemplo para cliente 123
cliente_exemplo = 123
recs = recomendar_para_cliente(cliente_exemplo)
print(f"Recomendações para cliente {cliente_exemplo}:")
for prod_id in recs:
    produto = df_produtos[df_produtos['id'] == prod_id]
    if not produto.empty:
        print(produto.iloc[0]['nome'])

Recomendações para cliente 123:
Óleo 80
Fantasia 30
Jogo de Tabuleiro 94
Bateria 44
Estante 49


In [17]:
"""
ecommerce_bigdata_demo.py
Exemplo completo: gera dados fictícios, insere no Elasticsearch, processa com Spark,
faz buscas e recomendações simples.
"""

import random
import time
import json
from datetime import datetime, timedelta

import pandas as pd

# Elasticsearch client
from elasticsearch import Elasticsearch, helpers

# PySpark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, desc

# ---------------------------
# Configurações
# ---------------------------
ES_HOST = "localhost"
ES_PORT = 9200
INDEX_NAME = "ecommerce"
BULK_CHUNK = 500  # tamanho do chunk para bulk insert

# ---------------------------
# 1) Gerar dados fictícios
# ---------------------------
def generate_products():
    produtos = [
        {"id": 1, "nome": "Notebook Dell", "categoria": "Eletrônicos", "preco": 3500.0, "marca": "Dell"},
        {"id": 2, "nome": "Smartphone Samsung", "categoria": "Eletrônicos", "preco": 2500.0, "marca": "Samsung"},
        {"id": 3, "nome": "Geladeira Brastemp", "categoria": "Eletrodomésticos", "preco": 1800.0, "marca": "Brastemp"},
        {"id": 4, "nome": "TV LG 50\"", "categoria": "Eletrônicos", "preco": 2800.0, "marca": "LG"},
        {"id": 5, "nome": "Cafeteira Nespresso", "categoria": "Eletrodomésticos", "preco": 600.0, "marca": "Nespresso"},
        {"id": 6, "nome": "Fone Bluetooth JBL", "categoria": "Eletrônicos", "preco": 350.0, "marca": "JBL"},
        {"id": 7, "nome": "Micro-ondas Electrolux", "categoria": "Eletrodomésticos", "preco": 700.0, "marca": "Electrolux"},
    ]
    return produtos

def generate_reviews(products, n_reviews=50):
    usuarios = [f"user_{i}" for i in range(1, 31)]
    comentarios = [
        "Excelente produto", "Bom custo-benefício", "Recomendo", "Não gostei", "Entrega rápida",
        "Qualidade razoável", "Muito satisfeito", "Poderia ser melhor"
    ]
    reviews = []
    for _ in range(n_reviews):
        p = random.choice(products)
        reviews.append({
            "produto_id": p["id"],
            "usuario": random.choice(usuarios),
            "nota": random.randint(1, 5),
            "comentario": random.choice(comentarios),
            "data": (datetime.now() - timedelta(days=random.randint(0, 365))).isoformat()
        })
    return reviews

def generate_transactions(products, n_tx=200):
    transacoes = []
    usuarios = [f"cliente_{i}" for i in range(1, 101)]
    for i in range(1, n_tx + 1):
        produto = random.choice(products)
        quantidade = random.randint(1, 4)
        valor_unit = produto["preco"] * random.uniform(0.8, 1.2)
        valor_total = round(valor_unit * quantidade, 2)
        transacoes.append({
            "transacao_id": i,
            "produto_id": produto["id"],
            "usuario": random.choice(usuarios),
            "quantidade": quantidade,
            "valor_total": valor_total,
            "data": (datetime.now() - timedelta(days=random.randint(0, 90))).isoformat()
        })
    return transacoes

# ---------------------------
# 2) Conectar e inserir no Elasticsearch
# ---------------------------
def connect_es(host=ES_HOST, port=ES_PORT):
    es = Elasticsearch([{"host": host, "port": port}])
    if not es.ping():
        raise ConnectionError(f"Não foi possível conectar ao Elasticsearch em {host}:{port}")
    return es

def create_index_if_not_exists(es, index_name=INDEX_NAME):
    # Mapeamento simples para demonstrar full-text e campos numéricos
    mapping = {
        "mappings": {
            "properties": {
                "nome": {"type": "text"},
                "categoria": {"type": "keyword"},
                "marca": {"type": "keyword"},
                "preco": {"type": "double"},
                "nota": {"type": "integer"},
                "comentario": {"type": "text"},
                "data": {"type": "date"},
                "valor_total": {"type": "double"},
                "quantidade": {"type": "integer"}
            }
        }
    }
    if not es.indices.exists(index=index_name):
        es.indices.create(index=index_name, body=mapping)

def bulk_insert_products(es, products, index_name=INDEX_NAME):
    actions = [
        {"_index": index_name, "_id": f"product_{p['id']}", "_source": p}
        for p in products
    ]
    helpers.bulk(es, actions, chunk_size=BULK_CHUNK)

def bulk_insert_reviews(es, reviews, index_name=INDEX_NAME):
    actions = [
        {"_index": index_name, "_source": r}
        for r in reviews
    ]
    helpers.bulk(es, actions, chunk_size=BULK_CHUNK)

def bulk_insert_transactions(es, transactions, index_name=INDEX_NAME):
    actions = [
        {"_index": index_name, "_source": t}
        for t in transactions
    ]
    helpers.bulk(es, actions, chunk_size=BULK_CHUNK)

# ---------------------------
# 3) Buscas com Query DSL (Exemplos)
# ---------------------------
def search_by_category(es, category, index_name=INDEX_NAME, size=10):
    body = {
        "query": {
            "match": {"categoria": category}
        },
        "size": size
    }
    res = es.search(index=index_name, body=body)
    return [hit["_source"] for hit in res["hits"]["hits"]]

def search_fulltext(es, text, index_name=INDEX_NAME, size=10):
    body = {
        "query": {
            "multi_match": {
                "query": text,
                "fields": ["nome^3", "comentario", "marca"]
            }
        },
        "size": size
    }
    res = es.search(index=index_name, body=body)
    return [hit["_source"] for hit in res["hits"]["hits"]]

# ---------------------------
# 4) Processamento com Spark
# ---------------------------
def spark_session(app_name="EcommerceSparkApp"):
    spark = SparkSession.builder \
        .appName(app_name) \
        .master("local[*]") \
        .config("spark.ui.showConsoleProgress", "false") \
        .getOrCreate()
    return spark

def analyze_transactions_with_spark(transactions):
    spark = spark_session()
    df = spark.createDataFrame(transactions)
    # Total de vendas por produto
    vendas_por_produto = df.groupBy("produto_id").agg(spark_sum("valor_total").alias("total_vendas"))
    vendas_por_produto = vendas_por_produto.orderBy(desc("total_vendas"))
    vendas_por_produto.show(truncate=False)
    # Produto mais vendido por quantidade
    mais_vendido = df.groupBy("produto_id").agg(spark_sum("quantidade").alias("qtd_vendida")).orderBy(desc("qtd_vendida"))
    mais_vendido.show(truncate=False)
    # Coletar resultados para uso posterior (pegar top 3)
    top3 = mais_vendido.limit(3).toPandas()
    spark.stop()
    return top3

# ---------------------------
# 5) Recomendação simples (combina vendas e avaliações)
# ---------------------------
def recommend_product(es, transactions, reviews, top_n=3):
    # Recomendação simples: produtos com maior soma de quantidade + média de nota
    df_tx = pd.DataFrame(transactions)
    df_rev = pd.DataFrame(reviews)
    vendas = df_tx.groupby("produto_id")["quantidade"].sum().rename("total_qtd")
    medias = df_rev.groupby("produto_id")["nota"].mean().rename("media_nota")
    score = pd.concat([vendas, medias], axis=1).fillna(0)
    # Normalizar e combinar
    score["score"] = (score["total_qtd"] / (score["total_qtd"].max() or 1)) * 0.7 + (score["media_nota"] / 5.0) * 0.3
    top = score.sort_values("score", ascending=False).head(top_n).reset_index()
    # Buscar detalhes no Elasticsearch
    recs = []
    for _, row in top.iterrows():
        prod = es.get(index=INDEX_NAME, id=f"product_{int(row['produto_id'])}")["_source"]
        recs.append({"produto": prod, "score": float(row["score"])})
    return recs

# ---------------------------
# 6) Fluxo principal
# ---------------------------
def main():
    print("Gerando dados fictícios...")
    products = generate_products()
    reviews = generate_reviews(products, n_reviews=120)
    transactions = generate_transactions(products, n_tx=800)

    print("Conectando ao Elasticsearch...")
    es = connect_es()

    print("Criando índice (se necessário) e inserindo dados...")
    create_index_if_not_exists(es, INDEX_NAME)
    # Inserir produtos com IDs estáveis
    bulk_insert_products(es, products)
    # Inserir reviews e transações (como documentos separados)
    bulk_insert_reviews(es, reviews)
    bulk_insert_transactions(es, transactions)

    print("Pausando brevemente para garantir indexação...")
    time.sleep(2)

    print("\nExemplo de busca por categoria 'Eletrônicos':")
    eletr = search_by_category(es, "Eletrônicos", size=5)
    for p in eletr:
        print(f" - {p.get('nome')} | R$ {p.get('preco')}")

    print("\nExemplo de busca full-text por 'Notebook':")
    ft = search_fulltext(es, "Notebook", size=5)
    for p in ft:
        print(f" - {p.get('nome')} | Categoria: {p.get('categoria')}")

    print("\nAnalisando transações com Spark (total vendas e mais vendidos)...")
    top3 = analyze_transactions_with_spark(transactions)
    print("\nTop 3 produtos por quantidade vendidos (Spark):")
    print(top3)

    print("\nGerando recomendações simples combinando vendas e avaliações...")
    recs = recommend_product(es, transactions, reviews, top_n=3)
    for r in recs:
        prod = r["produto"]
        print(f"Recomendado: {prod['nome']} (categoria: {prod['categoria']}) - score: {r['score']:.3f}")

    print("\nFluxo concluído.")

if __name__ == "__main__":
    main()


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/urllib3/connection.py", line 174, in _new_conn
    conn = connection.create_connection(
        (self._dns_host, self.port), self.timeout, **extra_kw
    )
  File "/usr/local/lib/python3.13/dist-packages/urllib3/util/connection.py", line 95, in create_connection
    raise err
  File "/usr/local/lib/python3.13/dist-packages/urllib3/util/connection.py", line 85, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
ConnectionRefusedError: [Errno 111] Connection refused

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/elasticsearch/connection/http_urllib3.py", line 255, in perform_request
    response = self.pool.urlopen(
        method, url, body, retries=Retry(False), headers=request_headers, **kw
    )
  File "/usr/local/lib/python3.13/dist-packages/urllib3/connectionpool.py", line 802, in url

Gerando dados fictícios...
Conectando ao Elasticsearch...


ConnectionError: Não foi possível conectar ao Elasticsearch em localhost:9200